In [61]:
import matplotlib.pyplot as plt
import pandas as pd
import pypsa
import numpy as np

# Load the datasets (ensure the paths match your local setup)
data_solar = pd.read_csv('data/pv_optimal.csv', sep=';', index_col=0, parse_dates=True)
data_wind = pd.read_csv('data/onshore_wind_1979-2017.csv', sep=';', index_col=0, parse_dates=True)
data_el = pd.read_csv('data/electricity_demand.csv', sep=';', index_col=0, parse_dates=True)

# Configuration from Part A
country = 'ESP'
year_cost = 2030 # Technology assumptions year

In [62]:
def get_costs(year):
    url = f"https://raw.githubusercontent.com/PyPSA/technology-data/v0.11.0/outputs/costs_{year}.csv"
    costs = pd.read_csv(url, index_col=[0, 1])
    
    # Unit conversion and defaults
    costs.loc[costs.unit.str.contains("/kW"), "value"] *= 1e3
    costs.unit = costs.unit.str.replace("/kW", "/MW")
    
    defaults = {"FOM": 0, "VOM": 0, "efficiency": 1, "fuel": 0, "investment": 0, "lifetime": 25, "discount rate": 0.07}
    costs = costs.value.unstack().fillna(defaults)
    costs.at["CCGT", "fuel"] = costs.at["gas", "fuel"]
    
    # Economics
    annuity = lambda r, n: r / (1.0 - 1.0 / (1.0 + r) ** n)
    costs["annuity"] = costs.apply(lambda x: annuity(x["discount rate"], x["lifetime"]), axis=1)
    costs["capital_cost"] = (costs["annuity"] + costs["FOM"] / 100) * costs["investment"]
    costs["marginal_cost"] = costs["VOM"] + costs["fuel"] / costs["efficiency"]
    
    return costs

costs_df = get_costs(year_cost)

In [ ]:
# Part b) Sensitivity Analysis Loop
simulation_years = range(1979, 2018)
results = []

for year in simulation_years:
    print(f"Optimizing for weather year: {year}...")
    
    n = pypsa.Network()
    
    # Create snapshots and remove timezone info
    snapshots = pd.date_range(start=f'{year}-01-01 00:00', 
                              end=f'{year}-12-31 23:00', 
                              freq='h').tz_localize(None)
    
    # Standardize to 8760 hours
    snapshots = snapshots[:8760]
    n.set_snapshots(snapshots)
    
    n.add("Bus", "Spain electricity")
    
    # Add Load - Ensuring values are float64
    demand_values = data_el[country].iloc[:8760].values.astype(float)
    n.add("Load", "demand", 
          bus="Spain electricity", 
          p_set=demand_values)
    
    # Add Conventional Generators
    for tech in ["coal", "CCGT"]:
        n.add("Generator", tech, bus="Spain electricity",
              p_nom_extendable=True,
              capital_cost=float(costs_df.at[tech, "capital_cost"]),
              marginal_cost=float(costs_df.at[tech, "marginal_cost"]))
        
    # Add Renewables
    # Filter weather data for the specific year and ensure float type
    wind_year = data_wind[data_wind.index.year == year][country].iloc[:8760].values.astype(float)
    solar_year = data_solar[data_solar.index.year == year][country].iloc[:8760].values.astype(float)

    n.add("Generator", "onwind", bus="Spain electricity",
          p_nom_extendable=True,
          capital_cost=float(costs_df.at["onwind", "capital_cost"]),
          marginal_cost=float(costs_df.at["onwind", "marginal_cost"]),
          p_max_pu=wind_year)

    n.add("Generator", "solar", bus="Spain electricity",
          p_nom_extendable=True,
          capital_cost=float(costs_df.at["solar", "capital_cost"]),
          marginal_cost=float(costs_df.at["solar", "marginal_cost"]),
          p_max_pu=solar_year)
    
    # Solve
    n.optimize(solver_name="highs")
    
    # Collect results (Converted to GW)
    caps = n.generators.p_nom_opt / 1e3
    caps.name = year
    results.append(caps)

# Combine into a single DataFrame for analysis
capacity_results = pd.concat(results, axis=1).T
print("Optimization Complete.")

Optimizing for weather year: 1979...


C:\Users\20221122\AppData\Local\Temp\ipykernel_58224\2697329775.py:52: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize(solver_name="highs")
Index(['Spain electricity'], dtype='str', name='name')


TypeError: Invalid array type: <class 'pandas.arrays.ArrowStringArray'>

In [ ]:
# Calculate mean and standard deviation across years
mean_capacities = capacity_results.mean()
std_capacities = capacity_results.std()

# Plotting
fig, ax = plt.subplots(figsize=(10, 6))
mean_capacities.plot(kind='bar', yerr=std_capacities, ax=ax, capsize=5, color=['indianred', 'yellowgreen', 'dodgerblue', 'gold'])

ax.set_title(f"Sensitivity of Optimal Capacities to Interannual Variability ({weather_years[0]}-{weather_years[-1]})")
ax.set_ylabel("Optimal Capacity (GW)")
ax.set_xlabel("Technology")
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# Display the raw data for analysis
print("Capacity Results (GW) per Year:")
print(capacity_results)